In [1]:
import copy
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge, HuberRegressor, BayesianRidge
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

from tqdm import tqdm
from tabulate import tabulate

warnings.filterwarnings('ignore')

In [2]:
DATA_DIR = Path("data/")
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else
                       "mps" if torch.backends.mps.is_available() else "cpu")

TARGET_COLS = [
    "Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %",
    "Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm",
]
DOSE_COL = "Массовая доля, %"
TEMP_COL = "Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C"
TIME_COL = "Время испытания | - Daimler Oxidation Test (DOT), ч"
BIO_COL = "Количество биотоплива | - Daimler Oxidation Test (DOT), % масс"
CAT_COL = "Дозировка катализатора, категория"

# --- Architecture ---
D_MODEL = 48
N_HEADS = 4
N_LAYERS = 2
D_FF = 96
DROPOUT = 0.20
N_SEEDS = 3

# --- Training ---
N_FOLDS = 5
EPOCHS = 1000
LR = 2e-4
WEIGHT_DECAY = 5e-3
PATIENCE = 120
BATCH_SIZE = 16
SEED = 42
N_ENSEMBLE_SEEDS = 5 

# --- Augmentation ---
AUG_NOISE_STD = 0.02
AUG_DROP_PROB = 0.05
AUG_DROP_THRESHOLD = 10.0
AUG_MULTIPLIER = 5
AUG_DOSE_JITTER = 0.015

# =====================================================================
# НАСТРОЙКА АНСАМБЛЯ (Для разных ноутбуков меняйте только эту строку)
# Варианты: 'bayesian_ridge', 'gpr', 'svr', 'ridge', None (только ST)
# =====================================================================
SECONDARY_MODEL = 'gpr' 

ST_WEIGHT = 0.85
SECONDARY_WEIGHT = 0.15 if SECONDARY_MODEL is not None else 0.0

preprocessing

In [3]:
def log_transform(y):
    return np.sign(y) * np.log1p(np.abs(y))

def log_inverse(y):
    return np.sign(y) * np.expm1(np.abs(y))

def extract_component_type(name: str) -> str:
    parts = name.rsplit("_", 1)
    return parts[0] if len(parts) == 2 and parts[1].isdigit() else name

def load_properties(path: str) -> dict:
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = ["Компонент", "Партия", "Показатель", "Единица", "Значение"]
    df["Значение"] = pd.to_numeric(df["Значение"], errors="coerce")
    props = {}
    for (comp, batch), grp in df.groupby(["Компонент", "Партия"]):
        d = {}
        for _, row in grp.iterrows():
            if pd.notna(row["Значение"]):
                d[row["Показатель"]] = row["Значение"]
        props[(comp, str(batch))] = d
    return props

def get_component_properties(comp, batch, props_dict, prop_names):
    key = (comp, str(batch))
    typical_key = (comp, "typical")
    batch_props = props_dict.get(key, {})
    typical_props = props_dict.get(typical_key, {})
    values, mask = [], []
    for pname in prop_names:
        if pname in batch_props:
            values.append(batch_props[pname])
            mask.append(1.0)
        elif pname in typical_props:
            values.append(typical_props[pname])
            mask.append(1.0)
        else:
            values.append(0.0)
            mask.append(0.0)
    return np.array(values, dtype=np.float32), np.array(mask, dtype=np.float32)

def select_numeric_properties(props_dict, min_coverage=0.03):
    all_keys = set()
    for d in props_dict.values():
        all_keys.update(d.keys())
    total = len(props_dict)
    selected = []
    for key in sorted(all_keys):
        count = sum(1 for d in props_dict.values() if key in d)
        if count / total >= min_coverage:
            selected.append(key)
    return selected

def build_component_vocab(train_df, test_df):
    all_comps = sorted(set(train_df["Компонент"].unique()) | set(test_df["Компонент"].unique()))
    comp_to_idx = {c: i + 1 for i, c in enumerate(all_comps)}
    return comp_to_idx

In [4]:
def build_scenarios(mixture_df, props_dict, prop_names, comp_type_to_idx, comp_to_idx, is_train=True):
    dot_condition_cols = [TEMP_COL, TIME_COL, BIO_COL, CAT_COL]
    n_types = len(comp_type_to_idx)
    scenarios = []

    for sid, grp in mixture_df.groupby("scenario_id"):
        comp_features, comp_ids, raw_doses = [], [], []

        for _, row in grp.iterrows():
            comp_name = row["Компонент"]
            batch = str(row["Наименование партии"]) if pd.notna(row["Наименование партии"]) else ""
            dose = row[DOSE_COL]
            raw_doses.append(dose)
            comp_ids.append(comp_to_idx.get(comp_name, 0))

            ctype = extract_component_type(comp_name)
            type_vec = np.zeros(n_types, dtype=np.float32)
            if ctype in comp_type_to_idx:
                type_vec[comp_type_to_idx[ctype]] = 1.0

            prop_vals, prop_mask = get_component_properties(comp_name, batch, props_dict, prop_names)
            dot_conds = np.array([row[c] for c in dot_condition_cols], dtype=np.float32)

            dose_x_temp = dose * dot_conds[0]
            dose_x_time = dose * dot_conds[1]
            dose_x_bio = dose * dot_conds[2]

            feat = np.concatenate([
                [dose], type_vec, prop_vals, prop_mask, dot_conds,
                [dose_x_temp, dose_x_time, dose_x_bio],
            ])
            comp_features.append(feat)

        scenario = {
            "components": np.stack(comp_features),
            "comp_ids": np.array(comp_ids, dtype=np.int64),
            "raw_doses": np.array(raw_doses, dtype=np.float32),
            "scenario_id": sid,
        }

        first_row = grp.iloc[0]
        scenario["strat_key"] = f"{int(first_row[TEMP_COL])}_{int(first_row[TIME_COL])}_{int(first_row[BIO_COL])}"

        if is_train:
            scenario["targets_orig"] = np.array([grp[TARGET_COLS[0]].iloc[0], grp[TARGET_COLS[1]].iloc[0]], dtype=np.float32)
            scenario["targets"] = np.array([log_transform(grp[TARGET_COLS[0]].iloc[0]), grp[TARGET_COLS[1]].iloc[0]], dtype=np.float32)
        scenarios.append(scenario)

    return scenarios

def build_handcrafted_features(scenarios, comp_type_to_idx, prop_names):
    n_types = len(comp_type_to_idx)
    feat_list = []

    for s in scenarios:
        comps = s["components"]
        n_comp = comps.shape[0]
        doses = comps[:, 0]
        type_onehots = comps[:, 1:1 + n_types]
        prop_start = 1 + n_types
        prop_end = prop_start + len(prop_names)
        properties = comps[:, prop_start:prop_end]
        mask_end = prop_end + len(prop_names)
        dot_conds = comps[0, mask_end:mask_end + 4]

        type_dose_sum = np.zeros(n_types, dtype=np.float32)
        type_dose_max = np.zeros(n_types, dtype=np.float32)
        type_dose_min = np.full(n_types, 999.0, dtype=np.float32)
        type_count = np.zeros(n_types, dtype=np.float32)
        
        for i in range(n_comp):
            t_idx = np.argmax(type_onehots[i])
            if type_onehots[i, t_idx] > 0:
                type_dose_sum[t_idx] += doses[i]
                type_dose_max[t_idx] = max(type_dose_max[t_idx], doses[i])
                type_dose_min[t_idx] = min(type_dose_min[t_idx], doses[i])
                type_count[t_idx] += 1
        type_dose_min[type_dose_min > 900] = 0.0

        total_dose = doses.sum() + 1e-8
        weighted_props = (properties * doses[:, None]).sum(axis=0) / total_dose
        prop_mask = comps[:, prop_end:mask_end]
        prop_coverage = prop_mask.mean(axis=0) 

        pairwise = []
        for i in range(n_types):
            for j in range(i + 1, n_types):
                pairwise.append(type_dose_sum[i] * type_dose_sum[j])
        pairwise = np.array(pairwise, dtype=np.float32)

        temp, time, bio, cat = dot_conds
        dot_x_type = []
        for i in range(n_types):
            dot_x_type.extend([type_dose_sum[i] * temp, type_dose_sum[i] * time, type_dose_sum[i] * bio])
        dot_x_type = np.array(dot_x_type, dtype=np.float32)

        cond_interactions = np.array([
            temp * time, temp * bio, time * bio, temp * time * bio, cat * temp, cat * bio,
        ], dtype=np.float32)

        general = np.array([
            n_comp, (type_count > 0).sum(), doses.sum(), doses.mean(), doses.std(), doses.max(), doses.min(), np.log1p(doses.sum()),
        ], dtype=np.float32)

        feat = np.concatenate([
            type_dose_sum, type_dose_max, type_dose_min, type_count, weighted_props,
            prop_coverage, pairwise, general, dot_conds, cond_interactions, dot_x_type,
        ])
        feat_list.append(feat)

    return np.stack(feat_list)

dataset

In [5]:
class DOTDataset(Dataset):
    def __init__(self, scenarios, feat_scaler=None, target_scaler=None, fit_scalers=False, augment=False):
        self.scenarios = scenarios
        self.augment = augment
        self.has_targets = "targets" in scenarios[0]

        all_feats = np.concatenate([s["components"] for s in scenarios], axis=0)

        if fit_scalers:
            self.feat_scaler = RobustScaler().fit(all_feats)
            if self.has_targets:
                all_targets = np.stack([s["targets"] for s in scenarios])
                self.target_scaler = RobustScaler().fit(all_targets)
            else:
                self.target_scaler = None
        else:
            self.feat_scaler = feat_scaler
            self.target_scaler = target_scaler

        for s in self.scenarios:
            s["components_scaled"] = self.feat_scaler.transform(
                s["components"]
            ).astype(np.float32)
            if self.has_targets and self.target_scaler is not None:
                s["targets_scaled"] = self.target_scaler.transform(
                    s["targets"].reshape(1, -1)
                ).flatten().astype(np.float32)

    def __len__(self):
        return len(self.scenarios) * (AUG_MULTIPLIER if self.augment else 1)

    def __getitem__(self, idx):
        real_idx = idx % len(self.scenarios)
        s = self.scenarios[real_idx]
        comps = s["components_scaled"].copy()
        comp_ids = s["comp_ids"].copy()

        if self.augment and idx >= len(self.scenarios):
            noise = np.random.randn(*comps.shape).astype(np.float32) * AUG_NOISE_STD
            comps = comps + noise

            # Dose jitter (first column)
            dose_noise = np.random.randn(comps.shape[0]).astype(np.float32) * AUG_DOSE_JITTER
            comps[:, 0] += dose_noise

            raw_doses = s.get("raw_doses", None)
            if raw_doses is not None and len(comps) > 4:
                keep_mask = np.ones(len(comps), dtype=bool)
                for i in range(len(comps)):
                    if raw_doses[i] < AUG_DROP_THRESHOLD and np.random.rand() < AUG_DROP_PROB:
                        keep_mask[i] = False
                if keep_mask.sum() >= 3:
                    comps = comps[keep_mask]
                    comp_ids = comp_ids[keep_mask]

        comps = torch.tensor(comps, dtype=torch.float32)
        comp_ids = torch.tensor(comp_ids, dtype=torch.long)
        if self.has_targets:
            targets = torch.tensor(s["targets_scaled"], dtype=torch.float32)
            return comps, comp_ids, targets
        return comps, comp_ids

def collate_fn(batch):
    has_targets = len(batch[0]) == 3
    if has_targets:
        components = [b[0] for b in batch]
        comp_ids = [b[1] for b in batch]
        targets = torch.stack([b[2] for b in batch])
    else:
        components = [b[0] for b in batch]
        comp_ids = [b[1] for b in batch]
        targets = None

    max_len = max(c.shape[0] for c in components)
    feat_dim = components[0].shape[1]
    padded = torch.zeros(len(components), max_len, feat_dim)
    padded_ids = torch.zeros(len(components), max_len, dtype=torch.long)
    mask = torch.zeros(len(components), max_len, dtype=torch.bool)
    for i, (c, cid) in enumerate(zip(components, comp_ids)):
        n = c.shape[0]
        padded[i, :n] = c
        padded_ids[i, :n] = cid
        mask[i, :n] = True

    if targets is not None:
        return padded, padded_ids, mask, targets
    return padded, padded_ids, mask

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        B, n_q, _ = Q.shape
        _, n_kv, _ = K.shape
        q = self.W_q(Q).view(B, n_q, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(K).view(B, n_kv, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(V).view(B, n_kv, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            mask_expanded = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(~mask_expanded, float("-inf"))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = torch.matmul(attn_weights, v)
        out = out.transpose(1, 2).contiguous().view(B, n_q, -1)
        out = self.W_o(out)
        return out, attn_weights

class SetAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, X, mask=None):
        attn_out, attn_weights = self.mha(X, X, X, mask)
        X = self.norm1(X + attn_out)
        X = self.norm2(X + self.ffn(X))
        return X, attn_weights

class PoolingByMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_seeds, dropout=0.1):
        super().__init__()
        self.seeds = nn.Parameter(torch.randn(1, n_seeds, d_model) * 0.01)
        self.mha = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, Z, mask=None):
        B = Z.shape[0]
        S = self.seeds.expand(B, -1, -1)
        out, attn_weights = self.mha(S, Z, Z, mask)
        out = self.norm(S + out)
        return out, attn_weights

class SetTransformerDOT(nn.Module):
    def __init__(self, feat_dim, n_components, d_model=48, n_heads=4,
                 n_layers=2, d_ff=96, n_seeds=3, dropout=0.25, comp_embed_dim=8):
        super().__init__()
        self.comp_embed_dim = comp_embed_dim
        self.comp_embedding = nn.Embedding(n_components + 1, comp_embed_dim, padding_idx=0)

        total_input_dim = feat_dim + comp_embed_dim

        self.input_proj = nn.Sequential(
            nn.Linear(total_input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.sab_layers = nn.ModuleList([
            SetAttentionBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.pma = PoolingByMultiHeadAttention(d_model, n_heads, n_seeds, dropout)

        self.head_visc = nn.Sequential(
            nn.Linear(d_model * n_seeds, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
        self.head_oxid = nn.Sequential(
            nn.Linear(d_model * n_seeds, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, comp_ids, mask=None):
        comp_emb = self.comp_embedding(comp_ids)
        h = torch.cat([x, comp_emb], dim=-1)

        h = self.input_proj(h)
        if mask is not None:
            h = h * mask.unsqueeze(-1).float()
        all_attn = []
        for sab in self.sab_layers:
            h, attn_w = sab(h, mask)
            all_attn.append(attn_w)
            if mask is not None:
                h = h * mask.unsqueeze(-1).float()
        pooled, pma_attn = self.pma(h, mask)
        all_attn.append(pma_attn)
        pooled_flat = pooled.view(pooled.shape[0], -1)

        pred_visc = self.head_visc(pooled_flat)
        pred_oxid = self.head_oxid(pooled_flat)
        pred = torch.cat([pred_visc, pred_oxid], dim=-1)
        return pred, all_attn

In [7]:
def weighted_huber_loss(pred, target, delta=1.0):
    diff = pred - target
    abs_diff = torch.abs(diff)
    huber = torch.where(abs_diff <= delta,
                        0.5 * diff ** 2,
                        delta * (abs_diff - 0.5 * delta))
    return huber.mean(dim=0).mean()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    count = 0
    for batch in loader:
        if len(batch) == 4:
            padded, comp_ids, mask, targets = batch
        else:
            continue
        padded, comp_ids, mask, targets = padded.to(device), comp_ids.to(device), mask.to(device), targets.to(device)
        optimizer.zero_grad()
        pred, _ = model(padded, comp_ids, mask)
        loss = weighted_huber_loss(pred, targets, delta=1.0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * padded.shape[0]
        count += padded.shape[0]
    return total_loss / count

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    count = 0
    preds, trues = [], []
    for batch in loader:
        if len(batch) == 4:
            padded, comp_ids, mask, targets = batch
        else:
            continue
        padded, comp_ids, mask, targets = padded.to(device), comp_ids.to(device), mask.to(device), targets.to(device)
        pred, _ = model(padded, comp_ids, mask)
        loss = weighted_huber_loss(pred, targets, delta=1.0)
        total_loss += loss.item() * padded.shape[0]
        count += padded.shape[0]
        preds.append(pred.cpu())
        trues.append(targets.cpu())
    return total_loss / count, torch.cat(preds), torch.cat(trues)

def get_strat_labels(scenarios):
    strat_keys = [s["strat_key"] for s in scenarios]
    unique_keys = sorted(set(strat_keys))
    key_to_idx = {k: i for i, k in enumerate(unique_keys)}
    return np.array([key_to_idx[k] for k in strat_keys])

In [8]:
class SecondaryModelWrapper:
    """
    Обертка для классического ML. Поддерживает вероятностные подходы.
    """
    def __init__(self, model_type='bayesian_ridge'):
        self.model_type = model_type
        self.best_visc_model = None
        self.best_oxid_model = None

    def fit(self, X_train, y_train_visc, y_train_oxid, X_val, y_val_orig, y_scaler_visc, y_scaler_oxid):
        if self.model_type is None:
            return

        # 1. RIDGE (Оставил на всякий случай как бейзлайн)
        if self.model_type == 'ridge':
            best_visc_mae = float('inf')
            for alpha in [0.1, 1.0, 5.0, 10.0, 50.0]:
                m = Ridge(alpha=alpha)
                m.fit(X_train, y_train_visc)
                p = y_scaler_visc.inverse_transform(m.predict(X_val).reshape(-1, 1)).ravel()
                mae = np.abs(log_inverse(p) - y_val_orig[:, 0]).mean()
                if mae < best_visc_mae:
                    best_visc_mae = mae
                    self.best_visc_model = m

            best_oxid_mae = float('inf')
            for alpha in [0.1, 1.0, 5.0, 10.0, 50.0]:
                m = Ridge(alpha=alpha)
                m.fit(X_train, y_train_oxid)
                p = y_scaler_oxid.inverse_transform(m.predict(X_val).reshape(-1, 1)).ravel()
                mae = np.abs(p - y_val_orig[:, 1]).mean()
                if mae < best_oxid_mae:
                    best_oxid_mae = mae
                    self.best_oxid_model = m

        # 2. BAYESIAN RIDGE (Байесовская гребневая регрессия)
        elif self.model_type == 'bayesian_ridge':
            # BayesianRidge сам подбирает параметры регуляризации
            self.best_visc_model = BayesianRidge()
            self.best_visc_model.fit(X_train, y_train_visc)
            
            self.best_oxid_model = BayesianRidge()
            self.best_oxid_model.fit(X_train, y_train_oxid)

        # 3. SVR (Support Vector Regression с RBF ядром)
        elif self.model_type == 'svr':
            best_visc_mae = float('inf')
            for C in [1.0, 10.0, 50.0]:
                for eps in [0.01, 0.1, 0.5]:
                    m = SVR(kernel='rbf', C=C, epsilon=eps)
                    m.fit(X_train, y_train_visc)
                    p = y_scaler_visc.inverse_transform(m.predict(X_val).reshape(-1, 1)).ravel()
                    mae = np.abs(log_inverse(p) - y_val_orig[:, 0]).mean()
                    if mae < best_visc_mae:
                        best_visc_mae = mae
                        self.best_visc_model = m

            best_oxid_mae = float('inf')
            for C in [1.0, 10.0, 50.0]:
                for eps in [0.01, 0.1, 0.5]:
                    m = SVR(kernel='rbf', C=C, epsilon=eps)
                    m.fit(X_train, y_train_oxid)
                    p = y_scaler_oxid.inverse_transform(m.predict(X_val).reshape(-1, 1)).ravel()
                    mae = np.abs(p - y_val_orig[:, 1]).mean()
                    if mae < best_oxid_mae:
                        best_oxid_mae = mae
                        self.best_oxid_model = m

        # 4. GAUSSIAN PROCESS REGRESSOR (Гауссовские процессы)
        elif self.model_type == 'gpr':
            # Используем ядро Matern + WhiteKernel (шум)
            kernel_visc = Matern(nu=1.5) + WhiteKernel(noise_level=1)
            self.best_visc_model = GaussianProcessRegressor(
                kernel=kernel_visc, n_restarts_optimizer=5, random_state=SEED
            )
            self.best_visc_model.fit(X_train, y_train_visc)

            kernel_oxid = Matern(nu=1.5) + WhiteKernel(noise_level=1)
            self.best_oxid_model = GaussianProcessRegressor(
                kernel=kernel_oxid, n_restarts_optimizer=5, random_state=SEED
            )
            self.best_oxid_model.fit(X_train, y_train_oxid)
            
        else:
            raise ValueError(f"Unknown model_type: {self.model_type}")

    def predict(self, X, y_scaler_visc, y_scaler_oxid):
        if self.model_type is None:
            return np.zeros((X.shape[0], 2))
            
        pred_visc = y_scaler_visc.inverse_transform(self.best_visc_model.predict(X).reshape(-1, 1)).ravel()
        pred_oxid = y_scaler_oxid.inverse_transform(self.best_oxid_model.predict(X).reshape(-1, 1)).ravel()
        pred_orig = np.column_stack([log_inverse(pred_visc), pred_oxid])
        return pred_orig

    def predict_with_std(self, X, y_scaler_visc, y_scaler_oxid):
        """
        Специальный метод для извлечения неуверенности (доверительных интервалов).
        Работает для 'gpr' и 'bayesian_ridge'.
        Возвращает: (predictions, std_visc, std_oxid)
        """
        if self.model_type not in ['gpr', 'bayesian_ridge']:
            raise ValueError("Standard deviation is only supported for GPR and Bayesian Ridge.")
            
        pred_visc_s, std_visc_s = self.best_visc_model.predict(X, return_std=True)
        pred_oxid_s, std_oxid_s = self.best_oxid_model.predict(X, return_std=True)
        
        # Обратное преобразование предиктов
        pred_visc = y_scaler_visc.inverse_transform(pred_visc_s.reshape(-1, 1)).ravel()
        pred_oxid = y_scaler_oxid.inverse_transform(pred_oxid_s.reshape(-1, 1)).ravel()
        pred_orig = np.column_stack([log_inverse(pred_visc), pred_oxid])
        
        # Масштабируем стандартное отклонение обратно
        std_visc = std_visc_s * y_scaler_visc.scale_[0]
        std_oxid = std_oxid_s * y_scaler_oxid.scale_[0]
        
        return pred_orig, std_visc, std_oxid

In [9]:
torch.manual_seed(SEED)
np.random.seed(SEED)

print("=" * 60)
print(f"Set Transformer + Secondary Model ({SECONDARY_MODEL})")
print("=" * 60)

train_df = pd.read_csv(DATA_DIR / "daimler_mixtures_train.csv", encoding="utf-8-sig")
test_df = pd.read_csv(DATA_DIR / "daimler_mixtures_test.csv", encoding="utf-8-sig")
props_dict = load_properties(DATA_DIR / "daimler_component_properties.csv")

all_comps = pd.concat([train_df["Компонент"], test_df["Компонент"]])
comp_types = sorted(set(all_comps.map(extract_component_type)))
comp_type_to_idx = {t: i for i, t in enumerate(comp_types)}
prop_names = select_numeric_properties(props_dict, min_coverage=0.03)
comp_to_idx = build_component_vocab(train_df, test_df)
n_components = len(comp_to_idx)

train_scenarios = build_scenarios(train_df, props_dict, prop_names, comp_type_to_idx, comp_to_idx, is_train=True)
test_scenarios = build_scenarios(test_df, props_dict, prop_names, comp_type_to_idx, comp_to_idx, is_train=False)
feat_dim = train_scenarios[0]["components"].shape[1]

hc_train = build_handcrafted_features(train_scenarios, comp_type_to_idx, prop_names)
hc_test = build_handcrafted_features(test_scenarios, comp_type_to_idx, prop_names)

y_train_targets = np.stack([s["targets"] for s in train_scenarios])
strat_labels = get_strat_labels(train_scenarios)

label_counts = pd.Series(strat_labels).value_counts()
label_map = {l: (l if c >= N_FOLDS else label_counts.index[0]) for l, c in label_counts.items()}
merged_labels = np.array([label_map[l] for l in strat_labels])

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# --- Хранилища для метрик и моделей ---
fold_st_models, fold_sec_models, fold_scalers = [], [], []
cv_scores_st, cv_scores_sec, cv_scores_hybrid = [], [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(train_scenarios)), merged_labels)):
    print(f"\n" + "═" * 40)
    print(f"  FOLD {fold + 1}/{N_FOLDS}  | Train: {len(train_idx)}, Val: {len(val_idx)}")
    print("═" * 40)
    
    # 1. Данные для второй модели
    hc_tr, hc_val = hc_train[train_idx], hc_train[val_idx]
    y_tr, y_val = y_train_targets[train_idx], y_train_targets[val_idx]
    y_val_orig = np.stack([train_scenarios[i]["targets_orig"] for i in val_idx])

    hc_scaler = RobustScaler().fit(hc_tr)
    hc_tr_s, hc_val_s = hc_scaler.transform(hc_tr), hc_scaler.transform(hc_val)

    y_scaler_visc = RobustScaler().fit(y_tr[:, 0:1])
    y_scaler_oxid = RobustScaler().fit(y_tr[:, 1:2])
    y_tr_visc_s = y_scaler_visc.transform(y_tr[:, 0:1]).ravel()
    y_tr_oxid_s = y_scaler_oxid.transform(y_tr[:, 1:2]).ravel()

    # 2. Обучение второй модели
    if SECONDARY_MODEL is not None:
        print(f"[*] Обучение {SECONDARY_MODEL.upper()}...")
    sec_model = SecondaryModelWrapper(model_type=SECONDARY_MODEL)
    sec_model.fit(hc_tr_s, y_tr_visc_s, y_tr_oxid_s, hc_val_s, y_val_orig, y_scaler_visc, y_scaler_oxid)
    sec_pred_orig = sec_model.predict(hc_val_s, y_scaler_visc, y_scaler_oxid)
    
    if SECONDARY_MODEL is not None:
        mae_sec = np.abs(sec_pred_orig - y_val_orig).mean(axis=0)
        cv_scores_sec.append(mae_sec)
        print(f"    {SECONDARY_MODEL.upper()} MAE: Visc={mae_sec[0]:.3f}, Oxid={mae_sec[1]:.3f}")
    else:
        cv_scores_sec.append([0.0, 0.0])
        
    fold_sec_models.append((sec_model, hc_scaler, y_scaler_visc, y_scaler_oxid))

    # 3. Обучение Set Transformer (Ансамбль сидов)
    print(f"[*] Обучение SetTransformer ({N_ENSEMBLE_SEEDS} seeds)...")
    fold_st_preds = []
    fold_seed_models, fold_seed_scalers = [], []

    for seed_i in range(N_ENSEMBLE_SEEDS):
        s = SEED * 7 + fold * 1000 + seed_i * 137 + 13
        torch.manual_seed(s)
        np.random.seed(s)

        fold_train = [copy.deepcopy(train_scenarios[i]) for i in train_idx]
        fold_val = [copy.deepcopy(train_scenarios[i]) for i in val_idx]

        train_ds = DOTDataset(fold_train, fit_scalers=True, augment=True)
        val_ds = DOTDataset(fold_val, feat_scaler=train_ds.feat_scaler, target_scaler=train_ds.target_scaler, augment=False)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

        model = SetTransformerDOT(feat_dim=feat_dim, n_components=n_components, d_model=D_MODEL, n_heads=N_HEADS,
                                  n_layers=N_LAYERS, d_ff=D_FF, n_seeds=N_SEEDS, dropout=DROPOUT).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=2)

        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0
        best_epoch = 0

        # --- TQDM ПРОГРЕСС-БАР ---
        pbar = tqdm(range(1, EPOCHS + 1), desc=f"    Seed {seed_i+1}/{N_ENSEMBLE_SEEDS}", leave=False)
        
        for epoch in pbar:
            train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
            val_loss, val_preds, val_trues = evaluate(model, val_loader, DEVICE)
            scheduler.step()

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1

            # Обновляем текст справа от бара
            pbar.set_postfix({
                'Tr_loss': f"{train_loss:.3f}",
                'Val_loss': f"{val_loss:.3f}",
                'Best': f"{best_val_loss:.3f}",
                'Patience': f"{patience_counter}/{PATIENCE}"
            })

            if patience_counter >= PATIENCE:
                break
                
        pbar.close() # Закрываем бар
        print(f"    Seed {seed_i+1} завершен на {best_epoch} эпохе | Best Val Loss: {best_val_loss:.4f}")

        model.load_state_dict(best_state)
        model.eval()

        _, val_preds, val_trues = evaluate(model, val_loader, DEVICE)
        st_pred = train_ds.target_scaler.inverse_transform(val_preds.numpy())
        st_pred_orig = st_pred.copy()
        st_pred_orig[:, 0] = log_inverse(st_pred[:, 0])
        fold_st_preds.append(st_pred_orig)

        fold_seed_models.append(model)
        fold_seed_scalers.append((train_ds.feat_scaler, train_ds.target_scaler))

    # Усредняем предикты ST по сидам
    st_pred_avg = np.mean(fold_st_preds, axis=0)
    mae_st = np.abs(st_pred_avg - y_val_orig).mean(axis=0)
    cv_scores_st.append(mae_st)
    print(f"    ST Ensemble MAE: Visc={mae_st[0]:.3f}, Oxid={mae_st[1]:.3f}")

    # 4. Гибридный предикт
    hybrid_pred = ST_WEIGHT * st_pred_avg + SECONDARY_WEIGHT * sec_pred_orig
    mae_hybrid = np.abs(hybrid_pred - y_val_orig).mean(axis=0)
    cv_scores_hybrid.append(mae_hybrid)
    print(f"    HYBRID MAE:      Visc={mae_hybrid[0]:.3f}, Oxid={mae_hybrid[1]:.3f}")

    fold_st_models.append((fold_seed_models, fold_seed_scalers))

Set Transformer + Secondary Model (gpr)

════════════════════════════════════════
  FOLD 1/5  | Train: 133, Val: 34
════════════════════════════════════════
[*] Обучение GPR...
    GPR MAE: Visc=85.732, Oxid=16.164
[*] Обучение SetTransformer (5 seeds)...


    Seed 1 завершен на 22 эпохе | Best Val Loss: 0.0623


    Seed 2 завершен на 30 эпохе | Best Val Loss: 0.0548


    Seed 3 завершен на 18 эпохе | Best Val Loss: 0.0567


    Seed 4 завершен на 48 эпохе | Best Val Loss: 0.0618


...

    Seed 1 завершен на 165 эпохе | Best Val Loss: 0.0552


    Seed 2 завершен на 126 эпохе | Best Val Loss: 0.0561


    Seed 3 завершен на 173 эпохе | Best Val Loss: 0.0700


    Seed 4 завершен на 16 эпохе | Best Val Loss: 0.0701


    Seed 5 завершен на 143 эпохе | Best Val Loss: 0.0693
    ST Ensemble MAE: Visc=32.923, Oxid=14.443
    HYBRID MAE:      Visc=31.337, Oxid=14.467


In [10]:
# --- Предикт на тестовых данных ---
all_st_preds = []
all_sec_preds = []

# Предикты второй модели
for sec_model, hc_sc, y_sc_visc, y_sc_oxid in fold_sec_models:
    hc_test_s = hc_sc.transform(hc_test)
    pred_orig = sec_model.predict(hc_test_s, y_sc_visc, y_sc_oxid)
    all_sec_preds.append(pred_orig)

sec_ensemble = np.mean(all_sec_preds, axis=0)

# Предикты Set Transformer
for fold_models, fold_scaler_list in fold_st_models:
    for model, (feat_sc, tgt_sc) in zip(fold_models, fold_scaler_list):
        test_copy = []
        for s in test_scenarios:
            sc = {
                "components": feat_sc.transform(s["components"]).astype(np.float32),
                "comp_ids": s["comp_ids"],
                "scenario_id": s["scenario_id"],
            }
            test_copy.append(sc)

        test_comps = [torch.tensor(s["components"], dtype=torch.float32) for s in test_copy]
        test_ids = [torch.tensor(s["comp_ids"], dtype=torch.long) for s in test_copy]
        max_len = max(t.shape[0] for t in test_comps)
        feat_d = test_comps[0].shape[1]
        padded = torch.zeros(len(test_comps), max_len, feat_d)
        padded_ids = torch.zeros(len(test_comps), max_len, dtype=torch.long)
        mask_t = torch.zeros(len(test_comps), max_len, dtype=torch.bool)
        for i, (t, tid) in enumerate(zip(test_comps, test_ids)):
            n = t.shape[0]
            padded[i, :n] = t
            padded_ids[i, :n] = tid
            mask_t[i, :n] = True

        with torch.no_grad():
            model.eval()
            pred, _ = model(padded.to(DEVICE), padded_ids.to(DEVICE), mask_t.to(DEVICE))
            pred = tgt_sc.inverse_transform(pred.cpu().numpy())
            pred_orig = pred.copy()
            pred_orig[:, 0] = log_inverse(pred[:, 0])
            all_st_preds.append(pred_orig)

st_ensemble = np.mean(all_st_preds, axis=0)
hybrid_ensemble = ST_WEIGHT * st_ensemble + SECONDARY_WEIGHT * sec_ensemble

# Сохранение
test_sids = [s["scenario_id"] for s in test_scenarios]
pred_df = pd.DataFrame({
    "scenario_id": test_sids,
    TARGET_COLS[0]: hybrid_ensemble[:, 0],
    TARGET_COLS[1]: hybrid_ensemble[:, 1],
})
pred_df.to_csv("results/predictions_st_gpr.csv", index=False)


# ==========================================
# КРАСИВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ПО ФОЛДАМ
# ==========================================
cv_st = np.array(cv_scores_st)
cv_sec = np.array(cv_scores_sec)
cv_hybrid = np.array(cv_scores_hybrid)

table_data = []
for i in range(N_FOLDS):
    row = [
        f"Fold {i+1}",
        f"{cv_st[i, 0]:.3f}", f"{cv_st[i, 1]:.3f}",
        f"{cv_sec[i, 0]:.3f}" if SECONDARY_MODEL else "-", 
        f"{cv_sec[i, 1]:.3f}" if SECONDARY_MODEL else "-",
        f"{cv_hybrid[i, 0]:.3f}", f"{cv_hybrid[i, 1]:.3f}"
    ]
    table_data.append(row)

# Добавляем строку со средними значениями
mean_st = cv_st.mean(axis=0)
std_st = cv_st.std(axis=0)
mean_sec = cv_sec.mean(axis=0)
std_sec = cv_sec.std(axis=0)
mean_hyb = cv_hybrid.mean(axis=0)
std_hyb = cv_hybrid.std(axis=0)

table_data.append(["-"*6, "-"*9, "-"*9, "-"*9, "-"*9, "-"*9, "-"*9])
table_data.append([
    "MEAN ± STD",
    f"{mean_st[0]:.3f} ± {std_st[0]:.3f}", f"{mean_st[1]:.3f} ± {std_st[1]:.3f}",
    f"{mean_sec[0]:.3f} ± {std_sec[0]:.3f}" if SECONDARY_MODEL else "-", 
    f"{mean_sec[1]:.3f} ± {std_sec[1]:.3f}" if SECONDARY_MODEL else "-",
    f"{mean_hyb[0]:.3f} ± {std_hyb[0]:.3f}", f"{mean_hyb[1]:.3f} ± {std_hyb[1]:.3f}"
])

headers = ["Fold", "ST (Visc)", "ST (Oxid)", f"{str(SECONDARY_MODEL).upper()} (Visc)", f"{str(SECONDARY_MODEL).upper()} (Oxid)", "Hybrid (Visc)", "Hybrid (Oxid)"]

print("\n" + "="*80)
print(f" ИТОГОВЫЕ МЕТРИКИ КРОСС-ВАЛИДАЦИИ (MAE) | Веса: ST={ST_WEIGHT}, {SECONDARY_MODEL}={SECONDARY_WEIGHT}")
print("="*80)
print(tabulate(table_data, headers=headers, tablefmt="github", stralign="center"))
print("="*80)
print(f"Predictions saved to 'predictions.csv'. Total records: {len(pred_df)}")


 ИТОГОВЫЕ МЕТРИКИ КРОСС-ВАЛИДАЦИИ (MAE) | Веса: ST=0.85, gpr=0.15
|    Fold    |    ST (Visc)    |   ST (Oxid)    |   GPR (Visc)    |   GPR (Oxid)   |  Hybrid (Visc)  |  Hybrid (Oxid)  |
|------------|-----------------|----------------|-----------------|----------------|-----------------|-----------------|
|   Fold 1   |     94.117      |     11.470     |     85.732      |     16.164     |     92.644      |     12.024      |
|   Fold 2   |     30.931      |     14.296     |     38.958      |     19.019     |     27.401      |     14.012      |
|   Fold 3   |     34.629      |     15.271     |     41.336      |     16.615     |     35.287      |     15.113      |
|   Fold 4   |     38.063      |     15.408     |     64.028      |     18.635     |     36.435      |     15.532      |
|   Fold 5   |     32.923      |     14.443     |     34.111      |     17.838     |     31.337      |     14.467      |
|   ------   |    ---------    |   ---------    |    ---------    |   ---------    |  